# pChEMBL Prediction v2: ECFP4 + Scaffold Split + HistGradientBoosting

**Major improvements over v1:**
- ✅ **Proper features**: ECFP4 fingerprints + clean descriptors (not ADMET noise)
- ✅ **Scaffold split**: Bemis-Murcko for realistic generalization (not random)
- ✅ **Faster models**: HistGradientBoostingRegressor (10-20x faster than GradientBoosting)
- ✅ **No scaling**: Tree models don't need it
- ✅ **Sparse storage**: Efficient fingerprint handling

**Expected performance:**
- **v1 (broken)**: R² ≈ 0.20, RMSE ≈ 1.09 (ADMET features + random split)
- **v2 (proper)**: R² ≈ 0.55-0.70, RMSE ≈ 0.60-0.75 (ECFP4 + scaffold split)

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.sparse import load_npz
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully")

## 2. Load Data (Descriptors + Fingerprints)

In [ ]:
# Load descriptors + metadata + scaffolds
csv_path = Path("../chembl_curated_out/molecules_42k_ecfp4_features.csv")
df = pd.read_csv(csv_path)

# Load fingerprints (use dense for now, can switch to sparse later)
fp_path = Path("../chembl_curated_out/fingerprints_ecfp4_dense.npy")
fingerprints = np.load(fp_path)

print(f"✅ Data loaded successfully")
print(f"   Descriptors + metadata: {df.shape}")
print(f"   Fingerprints: {fingerprints.shape}")
print(f"\n📊 Dataset overview:")
print(f"   Total molecules: {len(df):,}")
print(f"   Unique scaffolds: {df['scaffold'].nunique():,}")
print(f"\n🎯 Target variable (pChEMBL):")
print(f"   Mean: {df['pchembl_mean'].mean():.3f} ± {df['pchembl_mean'].std():.3f}")
print(f"   Range: [{df['pchembl_mean'].min():.2f}, {df['pchembl_mean'].max():.2f}]")

## 3. Prepare Features and Target

**Feature structure:**
- Fingerprints: 2048 ECFP4 bits (uint8)
- Descriptors: 26 RDKit physchem properties (float)

In [ ]:
# Define descriptor columns (exclude metadata)
metadata_cols = ['molecule_chembl_id', 'canonical_smiles', 'pchembl_mean', 'scaffold']
descriptor_cols = [col for col in df.columns if col not in metadata_cols]

# Extract descriptors
descriptors = df[descriptor_cols].values

# Combine fingerprints + descriptors
X_full = np.hstack([fingerprints, descriptors])
y_full = df['pchembl_mean'].values
scaffolds = df['scaffold'].values

print(f"✅ Features prepared:")
print(f"   Fingerprint features: {fingerprints.shape[1]}")
print(f"   Descriptor features: {len(descriptor_cols)}")
print(f"   Total features: {X_full.shape[1]}")
print(f"   Samples: {X_full.shape[0]:,}")
print(f"\n   Descriptor names: {descriptor_cols}")

# Critical validation checks
print(f"\n🔍 Data Validation Checks:")

# 1. Fingerprints should be binary (0/1)
fp_unique_vals = np.unique(fingerprints)
print(f"   1. Fingerprint values: {fp_unique_vals} (should be [0, 1])")
if not np.array_equal(fp_unique_vals, [0, 1]):
    print(f"      ⚠️  WARNING: Fingerprints not binary!")
else:
    print(f"      ✅ Fingerprints are binary (0/1)")

# 2. Check for NaNs in features
nan_count = np.isnan(X_full).sum()
print(f"   2. NaN values in features: {nan_count}")
if nan_count > 0:
    print(f"      ⚠️  WARNING: {nan_count} NaNs detected! Models may fail.")
else:
    print(f"      ✅ No NaNs in features")

# 3. Check for target leakage (columns with 'pchembl', 'activity', etc. in descriptors)
leakage_keywords = ['pchembl', 'activity', 'assay', 'target', 'bioactivity']
leakage_cols = [col for col in descriptor_cols if any(kw in col.lower() for kw in leakage_keywords)]
if leakage_cols:
    print(f"   3. ⚠️  WARNING: Potential target leakage detected in descriptors:")
    for col in leakage_cols:
        print(f"      - {col}")
else:
    print(f"   3. ✅ No obvious target leakage in descriptors")

# 4. Feature sparsity
fp_sparsity = (fingerprints == 0).sum() / fingerprints.size
print(f"   4. Fingerprint sparsity: {fp_sparsity:.2%} (expected: 85-95%)")
if fp_sparsity < 0.80 or fp_sparsity > 0.98:
    print(f"      ⚠️  WARNING: Unusual sparsity!")
else:
    print(f"      ✅ Sparsity looks normal")

# 5. Check for NaN scaffolds (GroupShuffleSplit can't handle NaN)
nan_scaffolds = pd.isna(scaffolds).sum()
print(f"   5. NaN scaffolds: {nan_scaffolds}")
if nan_scaffolds > 0:
    print(f"      ⚠️  Replacing NaN scaffolds with unique identifiers...")
    # Replace NaN scaffolds with unique IDs (each gets its own group)
    nan_mask = pd.isna(scaffolds)
    scaffolds_fixed = scaffolds.copy()
    scaffolds_fixed[nan_mask] = [f"unknown_scaffold_{i}" for i in range(nan_scaffolds)]
    scaffolds = scaffolds_fixed
    print(f"      ✅ Fixed {nan_scaffolds} NaN scaffolds")
else:
    print(f"      ✅ No NaN scaffolds")

## 4. Scaffold-Based Train/Val/Test Split (70/20/10)

**Why scaffold split?**
- Random split: Similar molecules in train/test → overly optimistic results
- Scaffold split: Different molecular scaffolds in train/test → realistic generalization
- Critical for binding affinity prediction (SAR is scaffold-dependent)

In [ ]:
# Step 1: Split into train+val (90%) and test (10%)
splitter_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
train_val_idx, test_idx = next(splitter_test.split(X_full, y_full, groups=scaffolds))

# Step 2: Split train+val into train (70%) and val (20%)
X_train_val = X_full[train_val_idx]
y_train_val = y_full[train_val_idx]
scaffolds_train_val = scaffolds[train_val_idx]

splitter_val = GroupShuffleSplit(n_splits=1, test_size=0.222, random_state=42)  # 0.222 * 90% ≈ 20%
train_idx, val_idx = next(splitter_val.split(X_train_val, y_train_val, groups=scaffolds_train_val))

# Final splits
X_train = X_train_val[train_idx]
y_train = y_train_val[train_idx]
X_val = X_train_val[val_idx]
y_val = y_train_val[val_idx]
X_test = X_full[test_idx]
y_test = y_full[test_idx]

print(f"✅ Scaffold-based split complete:")
print(f"   Train: {len(X_train):,} samples ({len(X_train)/len(X_full)*100:.1f}%)")
print(f"   Val:   {len(X_val):,} samples ({len(X_val)/len(X_full)*100:.1f}%)")
print(f"   Test:  {len(X_test):,} samples ({len(X_test)/len(X_full)*100:.1f}%)")

# Check scaffold overlap (should be zero)
train_scaffolds = set(scaffolds[train_val_idx[train_idx]])
val_scaffolds = set(scaffolds[train_val_idx[val_idx]])
test_scaffolds = set(scaffolds[test_idx])

overlap_val = len(train_scaffolds & val_scaffolds)
overlap_test = len((train_scaffolds | val_scaffolds) & test_scaffolds)

print(f"\n🧬 Scaffold overlap check:")
print(f"   Train scaffolds: {len(train_scaffolds):,}")
print(f"   Val scaffolds: {len(val_scaffolds):,}")
print(f"   Test scaffolds: {len(test_scaffolds):,}")
print(f"   Train-Val overlap: {overlap_val} (should be 0)")
print(f"   Train/Val-Test overlap: {overlap_test} (should be 0)")

if overlap_val == 0 and overlap_test == 0:
    print(f"   ✅ Perfect scaffold separation!")
else:
    print(f"   ⚠️  Warning: Scaffold leakage detected")

## 5. Null Baseline (Always Predict Mean)

In [ ]:
# Null model: always predict train mean
y_train_mean = y_train.mean()
y_val_pred_null = np.full(len(y_val), y_train_mean)
y_test_pred_null = np.full(len(y_test), y_train_mean)

null_mae_val = mean_absolute_error(y_val, y_val_pred_null)
null_rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred_null))
null_r2_val = r2_score(y_val, y_val_pred_null)

null_mae_test = mean_absolute_error(y_test, y_test_pred_null)
null_rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_null))
null_r2_test = r2_score(y_test, y_test_pred_null)

print(f"🎯 Null Baseline (predict mean = {y_train_mean:.3f}):")
print(f"\n   Validation:")
print(f"      MAE:  {null_mae_val:.3f} pChEMBL units")
print(f"      RMSE: {null_rmse_val:.3f}")
print(f"      R²:   {null_r2_val:.3f} (by definition ≈ 0)")
print(f"\n   Test:")
print(f"      MAE:  {null_mae_test:.3f} pChEMBL units")
print(f"      RMSE: {null_rmse_test:.3f}")
print(f"      R²:   {null_r2_test:.3f}")
print(f"\n   ⚡ Your model MUST beat these numbers!")

## 6. Train Random Forest (Baseline)

**Hyperparameters:**
- n_estimators: 500 (more trees = better)
- max_depth: 16 (prevent overfitting)
- min_samples_leaf: 5 (regularization)
- max_features: 'sqrt' (√2074 ≈ 45 features per split for ECFP4 + descriptors)

In [ ]:
%%time

print("🌲 Training Random Forest...")

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=16,
    min_samples_split=5,
    min_samples_leaf=5,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rf_model.fit(X_train, y_train)

# Predictions
y_val_pred_rf = rf_model.predict(X_val)
y_test_pred_rf = rf_model.predict(X_test)

# Metrics
from scipy.stats import spearmanr

rf_mae_val = mean_absolute_error(y_val, y_val_pred_rf)
rf_rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred_rf))
rf_r2_val = r2_score(y_val, y_val_pred_rf)
rf_spearman_val = spearmanr(y_val, y_val_pred_rf).correlation

rf_mae_test = mean_absolute_error(y_test, y_test_pred_rf)
rf_rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))
rf_r2_test = r2_score(y_test, y_test_pred_rf)
rf_spearman_test = spearmanr(y_test, y_test_pred_rf).correlation

print(f"\n🌲 Random Forest Results:")
print(f"\n   Validation:")
print(f"      MAE:      {rf_mae_val:.3f} pChEMBL units (null: {null_mae_val:.3f})")
print(f"      RMSE:     {rf_rmse_val:.3f} (null: {null_rmse_val:.3f})")
print(f"      R²:       {rf_r2_val:.3f} (null: {null_r2_val:.3f})")
print(f"      Spearman: {rf_spearman_val:.3f} (ranking correlation)")
print(f"\n   Test:")
print(f"      MAE:      {rf_mae_test:.3f} pChEMBL units (null: {null_mae_test:.3f})")
print(f"      RMSE:     {rf_rmse_test:.3f} (null: {null_rmse_test:.3f})")
print(f"      R²:       {rf_r2_test:.3f} (null: {null_r2_test:.3f})")
print(f"      Spearman: {rf_spearman_test:.3f} (ranking correlation)")

# Improvement over null
improvement_val = (null_rmse_val - rf_rmse_val) / null_rmse_val * 100
improvement_test = (null_rmse_test - rf_rmse_test) / null_rmse_test * 100
print(f"\n   📈 RMSE improvement over null:")
print(f"      Val: {improvement_val:.1f}%")
print(f"      Test: {improvement_test:.1f}%")

## 7. Train HistGradientBoosting (Fast & Strong)

**Why HistGradientBoosting?**
- 10-20x faster than GradientBoostingRegressor
- Handles large datasets better
- Often better performance
- Native support for missing values
- Early stopping prevents overfitting

**Hyperparameters (optimized for sparse ECFP):**
- max_iter: 2000 (early stopping will prevent overfitting)
- max_depth: 3 (shallow trees)
- max_leaf_nodes: 31 (controls complexity better than depth alone)

- learning_rate: 0.03- early_stopping: True (stops when val score plateaus)

- min_samples_leaf: 20 (sweet spot for ChEMBL noise)- l2_regularization: 1.0 (helps with noisy labels)

## 6b. Train ExtraTrees (Fast Alternative)

**Why ExtraTrees?**
- Often better than RF on sparse ECFP fingerprints
- More randomization → better generalization on noisy labels
- Fast training (similar to RF)
- No max_depth limit → learns deep patterns

**Hyperparameters:**
- n_estimators: 1000 (extra trees benefit from more trees)
- max_depth: None (unrestricted)
- min_samples_leaf: 5
- max_features: 'sqrt'

In [ ]:
%%time

from sklearn.ensemble import ExtraTreesRegressor

print("🌳 Training ExtraTrees...")

et_model = ExtraTreesRegressor(
    n_estimators=1000,
    max_depth=None,
    min_samples_leaf=5,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

et_model.fit(X_train, y_train)

# Predictions
y_val_pred_et = et_model.predict(X_val)
y_test_pred_et = et_model.predict(X_test)

# Metrics
et_mae_val = mean_absolute_error(y_val, y_val_pred_et)
et_rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred_et))
et_r2_val = r2_score(y_val, y_val_pred_et)
et_spearman_val = spearmanr(y_val, y_val_pred_et).correlation

et_mae_test = mean_absolute_error(y_test, y_test_pred_et)
et_rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_et))
et_r2_test = r2_score(y_test, y_test_pred_et)
et_spearman_test = spearmanr(y_test, y_test_pred_et).correlation

print(f"\n🌳 ExtraTrees Results:")
print(f"\n   Validation:")
print(f"      MAE:      {et_mae_val:.3f} pChEMBL units (null: {null_mae_val:.3f})")
print(f"      RMSE:     {et_rmse_val:.3f} (null: {null_rmse_val:.3f})")
print(f"      R²:       {et_r2_val:.3f} (null: {null_r2_val:.3f})")
print(f"      Spearman: {et_spearman_val:.3f} (ranking correlation)")
print(f"\n   Test:")
print(f"      MAE:      {et_mae_test:.3f} pChEMBL units (null: {null_mae_test:.3f})")
print(f"      RMSE:     {et_rmse_test:.3f} (null: {null_rmse_test:.3f})")
print(f"      R²:       {et_r2_test:.3f} (null: {null_r2_test:.3f})")
print(f"      Spearman: {et_spearman_test:.3f} (ranking correlation)")

# Improvement over null
improvement_val = (null_rmse_val - et_rmse_val) / null_rmse_val * 100
improvement_test = (null_rmse_test - et_rmse_test) / null_rmse_test * 100
print(f"\n   📈 RMSE improvement over null:")
print(f"      Val: {improvement_val:.1f}%")
print(f"      Test: {improvement_test:.1f}%")

In [ ]:
%%time

print("🚀 Training HistGradientBoosting...")

hgb_model = HistGradientBoostingRegressor(
    max_iter=2000,
    learning_rate=0.03,
    max_depth=3,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    random_state=42,
    verbose=1
)

hgb_model.fit(X_train, y_train)

# Predictions
y_val_pred_hgb = hgb_model.predict(X_val)
y_test_pred_hgb = hgb_model.predict(X_test)

# Metrics
hgb_mae_val = mean_absolute_error(y_val, y_val_pred_hgb)
hgb_rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred_hgb))
hgb_r2_val = r2_score(y_val, y_val_pred_hgb)
hgb_spearman_val = spearmanr(y_val, y_val_pred_hgb).correlation

hgb_mae_test = mean_absolute_error(y_test, y_test_pred_hgb)
hgb_rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_hgb))
hgb_r2_test = r2_score(y_test, y_test_pred_hgb)
hgb_spearman_test = spearmanr(y_test, y_test_pred_hgb).correlation

print(f"\n🚀 HistGradientBoosting Results:")
print(f"\n   Validation:")
print(f"      MAE:      {hgb_mae_val:.3f} pChEMBL units (null: {null_mae_val:.3f})")
print(f"      RMSE:     {hgb_rmse_val:.3f} (null: {null_rmse_val:.3f})")
print(f"      R²:       {hgb_r2_val:.3f} (null: {null_r2_val:.3f})")
print(f"      Spearman: {hgb_spearman_val:.3f} (ranking correlation)")
print(f"\n   Test:")
print(f"      MAE:      {hgb_mae_test:.3f} pChEMBL units (null: {null_mae_test:.3f})")
print(f"      RMSE:     {hgb_rmse_test:.3f} (null: {null_rmse_test:.3f})")
print(f"      R²:       {hgb_r2_test:.3f} (null: {null_r2_test:.3f})")
print(f"      Spearman: {hgb_spearman_test:.3f} (ranking correlation)")

# Improvement over null
improvement_val = (null_rmse_val - hgb_rmse_val) / null_rmse_val * 100
improvement_test = (null_rmse_test - hgb_rmse_test) / null_rmse_test * 100
print(f"\n   📈 RMSE improvement over null:")
print(f"      Val: {improvement_val:.1f}%")
print(f"      Test: {improvement_test:.1f}%")

## 8. Ensemble Model (Average All Models)

In [ ]:
# Ensemble: Average predictions from all three models
y_val_pred_ensemble = (y_val_pred_rf + y_val_pred_et + y_val_pred_hgb) / 3
y_test_pred_ensemble = (y_test_pred_rf + y_test_pred_et + y_test_pred_hgb) / 3

# Validation metrics
ens_mae_val = mean_absolute_error(y_val, y_val_pred_ensemble)
ens_rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred_ensemble))
ens_r2_val = r2_score(y_val, y_val_pred_ensemble)
ens_spearman_val = spearmanr(y_val, y_val_pred_ensemble).correlation

# Test metrics
ens_mae_test = mean_absolute_error(y_test, y_test_pred_ensemble)
ens_rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_ensemble))
ens_r2_test = r2_score(y_test, y_test_pred_ensemble)
ens_spearman_test = spearmanr(y_test, y_test_pred_ensemble).correlation

print(f"🎯 ENSEMBLE (RF + ET + HGB Average):")
print(f"\n   Validation:")
print(f"      MAE:      {ens_mae_val:.3f} pChEMBL units (null: {null_mae_val:.3f})")
print(f"      RMSE:     {ens_rmse_val:.3f} (null: {null_rmse_val:.3f})")
print(f"      R²:       {ens_r2_val:.3f} (null: {null_r2_val:.3f})")
print(f"      Spearman: {ens_spearman_val:.3f} (ranking correlation)")
print(f"\n   Test:")
print(f"      MAE:      {ens_mae_test:.3f} pChEMBL units (null: {null_mae_test:.3f})")
print(f"      RMSE:     {ens_rmse_test:.3f} (null: {null_rmse_test:.3f})")
print(f"      R²:       {ens_r2_test:.3f} (null: {null_r2_test:.3f})")
print(f"      Spearman: {ens_spearman_test:.3f} (ranking correlation)")

# Improvement over null
improvement_val = (null_rmse_val - ens_rmse_val) / null_rmse_val * 100
improvement_test = (null_rmse_test - ens_rmse_test) / null_rmse_test * 100
print(f"\n   📈 RMSE improvement over null:")
print(f"      Val: {improvement_val:.1f}%")
print(f"      Test: {improvement_test:.1f}%")

## 9. Results Summary and Comparison

In [ ]:
# Create comparison table
results_df = pd.DataFrame({
    'Model': ['Null Baseline', 'Random Forest', 'ExtraTrees', 'HistGradientBoosting', 'Ensemble (RF+ET+HGB)'],
    'Val MAE': [null_mae_val, rf_mae_val, et_mae_val, hgb_mae_val, ens_mae_val],
    'Val RMSE': [null_rmse_val, rf_rmse_val, et_rmse_val, hgb_rmse_val, ens_rmse_val],
    'Val R²': [null_r2_val, rf_r2_val, et_r2_val, hgb_r2_val, ens_r2_val],
    'Val Spearman': [0.0, rf_spearman_val, et_spearman_val, hgb_spearman_val, ens_spearman_val],
    'Test MAE': [null_mae_test, rf_mae_test, et_mae_test, hgb_mae_test, ens_mae_test],
    'Test RMSE': [null_rmse_test, rf_rmse_test, et_rmse_test, hgb_rmse_test, ens_rmse_test],
    'Test R²': [null_r2_test, rf_r2_test, et_r2_test, hgb_r2_test, ens_r2_test],
    'Test Spearman': [0.0, rf_spearman_test, et_spearman_test, hgb_spearman_test, ens_spearman_test],
})

print("\n" + "="*100)
print("📊 FINAL RESULTS COMPARISON")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)

# Highlight best model
best_model_idx = results_df['Test R²'].iloc[1:].idxmax()
best_model = results_df.loc[best_model_idx, 'Model']
best_r2 = results_df.loc[best_model_idx, 'Test R²']
best_rmse = results_df.loc[best_model_idx, 'Test RMSE']

print(f"\n🏆 Best Model: {best_model}")
print(f"   Test R²: {best_r2:.3f}")
print(f"   Test RMSE: {best_rmse:.3f} pChEMBL units")

# Compare to v1 (broken model)
print(f"\n📈 Improvement over v1 (ADMET features + random split):")
v1_r2 = 0.20
v1_rmse = 1.09
r2_improvement = ((best_r2 - v1_r2) / (1 - v1_r2)) * 100  # % of remaining error explained
rmse_improvement = ((v1_rmse - best_rmse) / v1_rmse) * 100

print(f"   R² improvement: {v1_r2:.2f} → {best_r2:.2f} (+{r2_improvement:.1f}% of remaining error)")
print(f"   RMSE improvement: {v1_rmse:.2f} → {best_rmse:.2f} (-{rmse_improvement:.1f}%)")

## 10. Visualize Predictions

In [ ]:
# Plot actual vs predicted for test set
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

models_to_plot = [
    ('Random Forest', y_test_pred_rf, rf_r2_test, rf_rmse_test),
    ('ExtraTrees', y_test_pred_et, et_r2_test, et_rmse_test),
    ('HistGradientBoosting', y_test_pred_hgb, hgb_r2_test, hgb_rmse_test),
    ('Ensemble', y_test_pred_ensemble, ens_r2_test, ens_rmse_test)
]

for idx, (name, y_pred, r2, rmse) in enumerate(models_to_plot):
    ax = axes[idx]
    
    # Scatter plot
    ax.scatter(y_test, y_pred, alpha=0.5, s=10, color='steelblue')
    
    # Perfect prediction line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
    
    ax.set_xlabel('Actual pChEMBL', fontsize=12)
    ax.set_ylabel('Predicted pChEMBL', fontsize=12)
    ax.set_title(f'{name}\nR² = {r2:.3f}, RMSE = {rmse:.3f}', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Prediction quality on test set:")

print(f"   Points close to red line = good predictions")
print(f"   Scatter = prediction uncertainty")

## 11. Feature Importance (Random Forest)

In [ ]:
# Get feature importance from Random Forest
# Note: Fingerprint bits named ecfp4_bit_0 to ecfp4_bit_2047, then descriptors
feature_names = [f'ecfp4_bit_{i}' for i in range(2048)] + descriptor_cols
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Split into fingerprint bits vs descriptors
fp_importance = feature_importance[feature_importance['feature'].str.startswith('ecfp4')]
desc_importance = feature_importance[~feature_importance['feature'].str.startswith('ecfp4')]

print(f"📊 Feature Importance Summary:")
print(f"   Total fingerprint importance: {fp_importance['importance'].sum():.3f}")
print(f"   Total descriptor importance: {desc_importance['importance'].sum():.3f}")
print(f"\n   Top 10 ECFP4 bits:")
for idx, row in fp_importance.head(10).iterrows():
    print(f"      {row['feature']:<20} {row['importance']:.4f}")
print(f"\n   Top 10 Descriptors:")
for idx, row in desc_importance.head(10).iterrows():
    print(f"      {row['feature']:<20} {row['importance']:.4f}")

# Plot descriptor importance
plt.figure(figsize=(10, 8))
top_desc = desc_importance.head(20)
plt.barh(range(len(top_desc)), top_desc['importance'], color='steelblue')
plt.yticks(range(len(top_desc)), top_desc['feature'])
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 20 Descriptor Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Save Trained Models

In [ ]:
# Save models
output_dir = Path("../models/pchembl_predictor_v2")
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, output_dir / 'random_forest_model.pkl')
joblib.dump(et_model, output_dir / 'extratrees_model.pkl')
joblib.dump(hgb_model, output_dir / 'histgradientboosting_model.pkl')

# Save feature names
with open(output_dir / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_names))

# Save metrics
results_df.to_csv(output_dir / 'model_comparison.csv', index=False)

print(f"✅ Models saved to {output_dir}/")
print(f"   - random_forest_model.pkl")
print(f"   - extratrees_model.pkl")
print(f"   - histgradientboosting_model.pkl")
print(f"   - feature_names.txt")
print(f"   - model_comparison.csv")

print(f"\n🎉 Training complete!")
print(f"\n📊 Summary:")
print(f"   Training approach: Scaffold split (realistic generalization)")
print(f"   Features: ECFP4 (2048) + Clean descriptors (26) = 2074 total")
print(f"   Best model: {best_model}")
print(f"   Test R²: {best_r2:.3f}")
print(f"   Test RMSE: {best_rmse:.3f} pChEMBL units")
print(f"\n🚀 Ready for prediction on 157k molecules without bioactivity!")